In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import re
import time
from bs4 import BeautifulSoup
import pandas as pd
import requests


City = "Karachi"
MAX_WORKERS = 3

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0.0.0 Safari/537.36"
    )
}

session = requests.Session()
session.headers.update(headers)


def fetch_with_backoff(url, max_retries=5):
    """Fetches a URL with exponential backoff if rate-limited or blocked."""
    for attempt in range(max_retries):
        try:
            response = session.get(url, timeout=12)

            if response.status_code in [403, 429]:
                wait_time = (attempt + 1) * 10
                print(
                    f"\n[!] Blocked/Rate limited (HTTP {response.status_code}). "
                    f"Cooling down IP for {wait_time}s... (Attempt {attempt+1}/{max_retries})"
                )
                time.sleep(wait_time)
                continue

            if response.status_code == 200:
                return response
        except Exception:
            time.sleep(3)

    return None


def extract_detail_data(item):
    """Fetches and parses a single detail page concurrently."""
    link = item["link"]
    property_name = item["property_name"]

    try:
        page = fetch_with_backoff(link)
        if not page:
            return None

        dpageSoup = BeautifulSoup(page.content, "html.parser")

        # Location (from detail page)
        try:
            location = dpageSoup.select_one('span[aria-label="Location"]').text.strip()
        except:
            location = ""

        # Price
        try:
            price = dpageSoup.select_one("span._63ea997b").text.strip()
        except:
            price = ""

        # Area
        try:
            area = dpageSoup.select_one('span[aria-label="Area"]').text.strip()
        except:
            area = ""

        # Bedrooms
        try:
            bedRoom = (
                dpageSoup.select_one('span[aria-label="Beds"]')
                .text.strip()
                .split()[0]
            )
        except:
            bedRoom = ""

        # Bathrooms
        try:
            bathroom = (
                dpageSoup.select_one('span[aria-label="Baths"]')
                .text.strip()
                .split()[0]
            )
        except:
            bathroom = ""

        # kitchens
        try:
            k_tag = dpageSoup.select_one(
                'span._9121cbf9:-soup-contains("Kitchen")'
            )
            if k_tag:
                nums = re.findall(r"\d+", k_tag.text)
                kitchen = int(nums[0]) if nums else 1
            else:
                kitchen = 0
        except:
            kitchen = 0

        # floor
        try:
            floorNum = ""
            for span in dpageSoup.select('span._9121cbf9'):
                text = span.text.strip()
                if text.lower().startswith("floor:") and not text.lower().startswith("floors in building"):
                    parts = text.split(":")
                    if len(parts) > 1 and parts[-1].strip():
                        floorNum = parts[-1].strip()
                    break
        except:
            floorNum = ""

        # total floors in bulding
        try:
            el_tag = dpageSoup.select_one(
                'span._9121cbf9:-soup-contains("Floors in Building")'
            )
            if el_tag and ":" in el_tag.text:
                parts = el_tag.text.split(":")
                floorsInBuilding = parts[-1].strip() if len(parts) > 1 else ""
            else:
                floorsInBuilding = ""
        except:
            floorsInBuilding = ""

        # elevators
        try:
            el_tag = dpageSoup.select_one(
                'span._9121cbf9:-soup-contains("Elevators")'
            )
            if el_tag:
                nums = re.findall(r"\d+", el_tag.text)
                elevator = int(nums[0]) if nums else 1
            else:
                elevator = 0
        except:
            elevator = 0

        # parking
        try:
            p_tag = dpageSoup.select_one(
                'span._9121cbf9:-soup-contains("Parking Spaces")'
            )
            if p_tag:
                nums = re.findall(r"\d+", p_tag.text)
                parking = int(nums[0]) if nums else 1
            else:
                parking = 0
        except:
            parking = 0

        # servant quarter
        try:
            sq_tag = dpageSoup.select_one(
                'span._9121cbf9:-soup-contains("Servant Quarters")'
            )
            if sq_tag:
                nums = re.findall(r"\d+", sq_tag.text)
                servant_quarters = int(nums[0]) if nums else 1
            else:
                servant_quarters = 0
        except:
            servant_quarters = 0

        # Binary Features
        furnished = (
            1
            if dpageSoup.select_one('span._9121cbf9:-soup-contains("Furnished")')
            else 0
        )
        electricityBackup = (
            1
            if dpageSoup.select_one(
                'span._9121cbf9:-soup-contains("Electricity Backup")'
            )
            else 0
        )

        # built year
        try:
            el_tag = dpageSoup.select_one(
                'span._9121cbf9:-soup-contains("Built in year")'
            )
            if el_tag:
                years = re.findall(r"\d{4}", el_tag.text)
                builtYear = years[0] if years else ""
            else:
                builtYear = ""
        except:
            builtYear = ""

        return {
            "property_name": property_name,
            "link": link,
            "address": item["location"],
            "location": location,
            "price": price,
            "area": area,
            "bedrooms": bedRoom,
            "bathrooms": bathroom,
            "kitchen": kitchen,
            "floor": floorNum,
            "total_floors": floorsInBuilding,
            "parking": parking,
            "elevator": elevator,
            "servant_quarters": servant_quarters,
            "electricity_backup": electricityBackup,
            "furnished": furnished,
            "built_year": builtYear,
        }

    except Exception:
        return None


# --- Execution Loop ---
start_page = int(input("Enter page number to start from: "))
total_pages = int(input("How many pages to scrape: "))

end_page = start_page + total_pages
flats = []

try:
    for pageNumber in range(start_page, end_page):
        url = f"https://www.zameen.com/Flats_Apartments/Karachi-2-{pageNumber}.html"

        page = fetch_with_backoff(url)
        if not page:
            print(f"Skipping page {pageNumber} after repeated network blocks.")
            continue

        pageSoup = BeautifulSoup(page.content, "html.parser")
        listings = pageSoup.select_one("ul.e20beb46")

        if listings is None:
            print(f"No listings found on page {pageNumber}. Skipping...")
            continue

        items = []
        for soup in listings.select("li.a37d52f0"):
            try:
                property_name = soup.select_one("h2._36dfb99f").text.strip()
                link = soup.select_one("a.d870ae17")["href"]

                if link.startswith("/"):
                    link = "https://www.zameen.com" + link

                location = soup.select_one("div.db1aca2f").text.strip()

                items.append({
                    "property_name": property_name,
                    "link": link,
                    "location": location,
                })
            except:
                continue

        page_results = []
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [
                executor.submit(extract_detail_data, item) for item in items
            ]

            for future in as_completed(futures):
                result = future.result()
                if result:
                    page_results.append(result)

        flats.extend(page_results)

        print(
            f"Page {pageNumber} completed | Extracted {len(page_results)} items"
            f" | Cumulative Total: {len(flats)}"
        )
        time.sleep(2.5)

except Exception as e:
    print("Scraping stopped/interrupted:", e)

finally:
    output_dir = "/content/drive/MyDrive/Data/Karachi/Flats"
    os.makedirs(output_dir, exist_ok=True)

    df = pd.DataFrame(flats)
    if not df.empty:
        df = df.drop_duplicates(subset=["link"])

    path = os.path.join(
        output_dir, f"flats_{City}_data_{start_page}_to_{end_page-1}.csv"
    )
    df.to_csv(path, index=False)

    print("\n----------------------------------------")
    print(f"Saved {len(df)} properties successfully!")
    print(f"Output File Path: {path}")

Enter page number to start from: 1
How many pages to scrape: 480
Page 1 completed | Extracted 25 items | Cumulative Total: 25
Page 2 completed | Extracted 25 items | Cumulative Total: 50
Page 3 completed | Extracted 25 items | Cumulative Total: 75
Page 4 completed | Extracted 25 items | Cumulative Total: 100
Page 5 completed | Extracted 25 items | Cumulative Total: 125
Page 6 completed | Extracted 25 items | Cumulative Total: 150
Page 7 completed | Extracted 25 items | Cumulative Total: 175
Page 8 completed | Extracted 25 items | Cumulative Total: 200
Page 9 completed | Extracted 25 items | Cumulative Total: 225
Page 10 completed | Extracted 25 items | Cumulative Total: 250
Page 11 completed | Extracted 25 items | Cumulative Total: 275
Page 12 completed | Extracted 25 items | Cumulative Total: 300
Page 13 completed | Extracted 25 items | Cumulative Total: 325
Page 14 completed | Extracted 25 items | Cumulative Total: 350
Page 15 completed | Extracted 25 items | Cumulative Total: 375
Pa